# Final GBT: frozen model and incoming observations

This notebook uses the project gradient boosting experiment. Weights and hyperparameters were fitted using 2023-2024 only. Forecasts cover the next 96 quarter-hours; later observed history may enter the next request, without fitting. The first forecast below is produced **before loading any 2025 observations**.

The 2025 source contains January-November. Earlier project experiments have already inspected 2025, so it is excluded from this fit/tuning but is not a never-inspected research holdout.

In [1]:
import os
from pathlib import Path
import sys

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"
ROOT = Path.cwd()
if not (ROOT / "main.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import joblib
import numpy as np
import pandas as pd
from src.config import load_run_settings
from src.models.gradient_boosting.inference import load_forecaster, forecast
from src.models.gradient_boosting.process import load_sources, required_history

In [2]:
RUN = ROOT / "outputs/runs/20261113T123354683990Z_gradient-boosting-final-grid-load-15min-evaluation_395ad9bfa94a"
bundle = load_forecaster(RUN / "checkpoints/model.joblib")
experiment, _ = load_run_settings(ROOT / "configs/experiments/gradient_boosting.yaml")
weights_before = joblib.hash(bundle["forecaster"].estimator_)
print("Training cutoff:", bundle["training_cutoff"])
print("Training labels:", bundle["training_rows"])
print("Input features:", len(bundle["feature_columns"]))

Training cutoff: 2024-12-31T22:45:00+00:00
Training labels: 69500
Input features: 28


## First forecast using development history only

`training_history` contains the last 676 prepared intervals of 2024. Features combine effective 25h/169h historical lags with shared calendar encodings and known BW holidays. The configurable one-hour buffer assumes availability; historical weather is not a verified publication-vintage archive.

In [3]:
origin = pd.Timestamp(experiment["split"]["test_start"]).tz_convert("UTC")
first = forecast(bundle, bundle["training_history"], origin)
first["features"].to_csv(ROOT / "Dataset/processed/gbt_final_demo_features.csv", index_label="timestamp_utc")
display(first["features"].head())
display(first["prediction"].rename("forecast_mwh").head())

,load_lag_24h_delay_60m,load_lag_168h_delay_60m,temperature_lag_24h_delay_60m,temperature_lag_168h_delay_60m,wind_lag_24h_delay_60m,wind_lag_168h_delay_60m,dew_point_lag_24h_delay_60m,dew_point_lag_168h_delay_60m,price_lag_24h_delay_60m,price_lag_168h_delay_60m,...,calendar_day_of_month_sin,calendar_day_of_month_cos,calendar_day_of_year_sin,calendar_day_of_year_cos,calendar_week_of_year_sin,calendar_week_of_year_cos,calendar_month_of_year_sin,calendar_month_of_year_cos,calendar_year,is_holiday
2024-12-31 23:00:00+00:00,1499.00,1314.00,-2.3,2.2,2.2,2.1,-2.7,1.3,35.66,90.00,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,2025,1
2024-12-31 23:15:00+00:00,1475.50,1300.00,-2.4,2.1,2.2,1.9,-2.8,1.3,35.66,90.00,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,2025,1
2024-12-31 23:30:00+00:00,1464.75,1281.50,-2.5,2.0,2.2,2.2,-2.9,1.2,35.66,90.00,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,2025,1
2024-12-31 23:45:00+00:00,1438.75,1261.25,-2.5,2.0,2.2,2.2,-2.8,1.0,35.66,90.00,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,2025,1
2025-01-01 00:00:00+00:00,1438.75,1298.25,-2.6,1.9,2.2,2.2,-2.9,0.9,50.49,91.37,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,2025,1


2024-12-31 23:00:00+00:00    1408.201025
2024-12-31 23:15:00+00:00    1386.315440
2024-12-31 23:30:00+00:00    1376.016495
2024-12-31 23:45:00+00:00    1340.659626
2025-01-01 00:00:00+00:00    1340.433816
Freq: 15min, Name: forecast_mwh, dtype: float64

## Reveal outcomes after prediction, then advance one day

The loader aligns Berlin consumption/prices and fixed UTC+2 weather, and carries causal imputation state forward. It retains original labels for scoring. In this historical replay the file is available locally, but `forecast()` only receives observations preceding the next origin. A production caller supplies the corresponding newly arrived prepared block.

In [4]:
demo, actual, _, metadata = load_sources(experiment["data"], "demo", initial_state=bundle["imputation_state"])
comparison = pd.DataFrame({"actual_mwh": actual.reindex(first["prediction"].index), "forecast_mwh": first["prediction"]})
print("First-day MAE (MWh):", (comparison.actual_mwh - comparison.forecast_mwh).abs().mean())
display(comparison.head())
next_origin = origin + pd.Timedelta(hours=24)
incoming = demo.loc[demo.index < next_origin]
history = pd.concat([bundle["training_history"], incoming]).tail(required_history(bundle["dataset"]))
second = forecast(bundle, history, next_origin)
display(second["prediction"].rename("second_day_forecast_mwh").head())

First-day MAE (MWh): 112.87259848994897


,actual_mwh,forecast_mwh
2024-12-31 23:00:00+00:00,1387.25,1408.201025
2024-12-31 23:15:00+00:00,1369.75,1386.315440
2024-12-31 23:30:00+00:00,1387.50,1376.016495
2024-12-31 23:45:00+00:00,1350.75,1340.659626
2025-01-01 00:00:00+00:00,1343.50,1340.433816


2025-01-01 23:00:00+00:00    1363.294620
2025-01-01 23:15:00+00:00    1363.844752
2025-01-01 23:30:00+00:00    1325.104448
2025-01-01 23:45:00+00:00    1289.331720
2025-01-02 00:00:00+00:00    1275.427569
Freq: 15min, Name: second_day_forecast_mwh, dtype: float64

In [5]:
saved = pd.read_csv(RUN / "predictions/final_test.csv")
np.testing.assert_allclose(first["prediction"].to_numpy(), saved.prediction.iloc[:96].to_numpy())
np.testing.assert_allclose(second["prediction"].to_numpy(), saved.prediction.iloc[96:192].to_numpy())
assert weights_before == joblib.hash(bundle["forecaster"].estimator_)
print("Both forecasts reproduce the evaluation; learned weights are unchanged.")

Both forecasts reproduce the evaluation; learned weights are unchanged.


To retrain, run the command in the project README and point `RUN` to its evaluation directory. This notebook deliberately performs inference only. Features for development are separately exported as `Dataset/processed/gbt_final_development.csv`; this notebook exports 96 unlabeled first-demo feature rows.